In [1]:
!pip install torch torchvision opencv-python numpy pyyaml matplotlib tqdm



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir())


Current folder: C:\Users\Raul\Desktop\Facultate\An 3\Proiect SBC\3 - Autonomous Car Navigation System
Files here: ['.ipynb_checkpoints', 'Agreement Document', 'annotations.xml', 'archive 2.zip', 'arhiva 1', 'arhiva 11', 'Documentation.docx', 'images', 'Juypyter Lab', 'Library File', 'masks', 'proiectvar2.ipynb', 'Raspblock-master', 'Raspblock-master.zip', 'Raspblock_Annex.zip', 'roads_segmentation.csv']


In [ ]:
from pathlib import Path
import random
import yaml
import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =====================================================
# UTILITARE
# =====================================================

def read_yolo_labels(label_path: Path):
    if not label_path.exists():
        return []
    lines = label_path.read_text(encoding="utf-8").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) != 5:
            continue
        c = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:])
        out.append((c, x, y, w, h))
    return out


# =====================================================
# DATASET CUSTOM 
# =====================================================

class YoloGridDataset(Dataset):
    def __init__(self, images_dir: Path, labels_dir: Path, img_size=416, S=13, C=15):
        if not images_dir.exists():
            raise FileNotFoundError(f"Nu găsesc images_dir: {images_dir.resolve()}")
        if not labels_dir.exists():
            raise FileNotFoundError(f"Nu găsesc labels_dir: {labels_dir.resolve()}")

        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.img_size = img_size
        self.S = S
        self.C = C

        self.imgs = sorted([p for p in images_dir.iterdir()
                            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

        if len(self.imgs) == 0:
            raise RuntimeError(f"Folderul există dar nu conține imagini: {images_dir.resolve()}")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        lbl_path = self.labels_dir / (img_path.stem + ".txt")
        labels = read_yolo_labels(lbl_path)

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32) / 255.0
        img_t = torch.from_numpy(img).permute(2, 0, 1)

        target = torch.zeros((self.S, self.S, 5 + self.C), dtype=torch.float32)

        for (c, x, y, w, h) in labels:
            if c < 0 or c >= self.C:
                continue
            cx = min(self.S - 1, max(0, int(x * self.S)))
            cy = min(self.S - 1, max(0, int(y * self.S)))

            target[cy, cx, 0] = 1.0
            target[cy, cx, 1:5] = torch.tensor([x, y, w, h])
            target[cy, cx, 5 + c] = 1.0

        return img_t, target


# =====================================================
# MODEL CNN (IMPLEMENTARE PROPRIE)
# =====================================================

class TinyDetector(nn.Module):
    def __init__(self, C=15):
        super().__init__()
        out_ch = 5 + C

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 208

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 104

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 52

            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 26

            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 13
        )

        self.head = nn.Conv2d(256, out_ch, 1)

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        x = x.permute(0, 2, 3, 1)  # (B,13,13,5+C)
        return x


# =====================================================
# LOSS FUNCTION
# =====================================================

def detection_loss(pred, target, lambda_box=5.0, lambda_noobj=0.5):
    """
    pred, target: (B,S,S,5+C)
    - obj: BCE pe toate celulele (dar penalizare mai mică pe noobj)
    - box: MSE doar unde obj=1
    - cls: CrossEntropy doar unde obj=1
    """
    obj_pred = torch.sigmoid(pred[..., 0])
    box_pred = torch.sigmoid(pred[..., 1:5])
    cls_logits = pred[..., 5:]

    obj_t = target[..., 0]
    box_t = target[..., 1:5]
    cls_t = target[..., 5:]  

    bce = F.binary_cross_entropy(obj_pred, obj_t, reduction="none")
    obj_loss = (bce * obj_t + bce * (1 - obj_t) * lambda_noobj).mean()

    mask = obj_t.unsqueeze(-1)  # (B,S,S,1)
    box_loss = (((box_pred - box_t) ** 2) * mask).mean()

    C = cls_logits.shape[-1]
    cls_id = cls_t.argmax(dim=-1)  # (B,S,S)
    ce = F.cross_entropy(cls_logits.reshape(-1, C), cls_id.reshape(-1), reduction="none")
    ce = ce.reshape(obj_t.shape)   # (B,S,S)
    cls_loss = (ce * obj_t).sum() / (obj_t.sum() + 1e-9)

    return obj_loss + lambda_box * box_loss + cls_loss

# =====================================================
# MAIN TRAIN FUNCTION
# =====================================================

def main():
    base = Path.cwd()
    data_yaml = base / "arhiva 1" / "car" / "data.yaml"

    if not data_yaml.exists():
        raise FileNotFoundError(f"Nu găsesc data.yaml la: {data_yaml.resolve()}")

    data = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    C = int(data["nc"])

    train_img = base / "arhiva 1" / "car" / "train" / "images"
    train_lbl = base / "arhiva 1" / "car" / "train" / "labels"

    val_img   = base / "arhiva 1" / "car" / "valid" / "images"
    val_lbl   = base / "arhiva 1" / "car" / "valid" / "labels"

    #test
    test_img  = base / "arhiva 1" / "car" / "test" / "images"
    test_lbl  = base / "arhiva 1" / "car" / "test" / "labels"

    IMG_SIZE = 416
    S = 13
    EPOCHS = 2
    BATCH = 16
    LR = 1e-3

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    print("Train images:", train_img.resolve())
    print("Val images:  ", val_img.resolve())

    train_ds = YoloGridDataset(train_img, train_lbl, img_size=IMG_SIZE, S=S, C=C)
    val_ds   = YoloGridDataset(val_img, val_lbl, img_size=IMG_SIZE, S=S, C=C)

    train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
    val_ld   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)

    model = TinyDetector(C=C).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for imgs, targets in tqdm(train_ld, desc=f"Epoch {epoch}/{EPOCHS}"):
            imgs = imgs.to(device)
            targets = targets.to(device)

            pred = model(imgs)
            loss = detection_loss(pred, targets)

            opt.zero_grad()
            loss.backward()
            opt.step()

            losses.append(loss.item())

        print(f"Epoch {epoch}/{EPOCHS} - Loss: {float(np.mean(losses)):.4f}")

    print("Training finished!")


In [ ]:
def xywh_to_xyxy(x, y, w, h):
    x1 = x - w/2
    y1 = y - h/2
    x2 = x + w/2
    y2 = y + h/2
    return x1, y1, x2, y2

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - inter + 1e-9
    return inter / union

def nms(boxes, scores, iou_thresh=0.5):
    idxs = np.argsort(scores)[::-1]
    keep = []
    while len(idxs) > 0:
        i = idxs[0]
        keep.append(i)
        rest = idxs[1:]
        new_rest = []
        for j in rest:
            if iou_xyxy(boxes[i], boxes[j]) < iou_thresh:
                new_rest.append(j)
        idxs = np.array(new_rest, dtype=np.int64)
    return keep

@torch.no_grad()
def decode_predictions(pred, conf_thresh=0.25):
    """
    pred: (S,S,5+C) raw
    return: list (cls, conf, x1,y1,x2,y2) normalized
    """
    S = pred.shape[0]
    C = pred.shape[-1] - 5

    obj = torch.sigmoid(pred[..., 0])
    box = torch.sigmoid(pred[..., 1:5])
    cls_prob = torch.softmax(pred[..., 5:], dim=-1)

    out = []
    for cy in range(S):
        for cx in range(S):
            o = float(obj[cy, cx].item())
            if o < conf_thresh:
                continue
            c = int(torch.argmax(cls_prob[cy, cx]).item())
            p = float(cls_prob[cy, cx, c].item())
            conf = o * p

            x, y, w, h = box[cy, cx].tolist()
            x1, y1, x2, y2 = xywh_to_xyxy(x, y, w, h)
            x1 = max(0.0, min(1.0, x1))
            y1 = max(0.0, min(1.0, y1))
            x2 = max(0.0, min(1.0, x2))
            y2 = max(0.0, min(1.0, y2))
            out.append((c, conf, x1, y1, x2, y2))

    return out

def draw_dets(img_rgb, dets, names):
    h, w = img_rgb.shape[:2]
    img = img_rgb.copy()
    for c, conf, x1,y1,x2,y2 in dets:
        X1 = int(x1*w); Y1 = int(y1*h); X2 = int(x2*w); Y2 = int(y2*h)
        cv2.rectangle(img, (X1,Y1), (X2,Y2), (0,255,0), 2)
        txt = f"{names[c]} {conf:.2f}"
        cv2.putText(img, txt, (X1, max(15, Y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    return img

@torch.no_grad()
def evaluate_prf1(model, loader, device, C, iou_thresh=0.5, conf_thresh=0.25):
    """
    Metric simplu (nu mAP):
    - TP: predicție cu clasă corectă și IoU>=threshold cu un GT ne-matcheduit
    - FP: predicție fără match
    - FN: GT nematcheduit
    """
    model.eval()
    tp = fp = fn = 0

    for imgs, targets in tqdm(loader, desc="Eval", leave=False):
        imgs = imgs.to(device)
        preds = model(imgs).cpu()

        B = preds.shape[0]
        S = preds.shape[1]

        for b in range(B):
            # decode preds
            pred_list = decode_predictions(preds[b], conf_thresh=conf_thresh)

            # NMS
            if len(pred_list) > 0:
                boxes = np.array([[p[2],p[3],p[4],p[5]] for p in pred_list], dtype=np.float32)
                scores = np.array([p[1] for p in pred_list], dtype=np.float32)
                keep = nms(boxes, scores, iou_thresh=0.5)
                pred_list = [pred_list[i] for i in keep]

            # extract GT from grid targets
            gt = []
            t = targets[b]
            for cy in range(S):
                for cx in range(S):
                    if t[cy, cx, 0].item() == 1.0:
                        cls = int(torch.argmax(t[cy, cx, 5:]).item())
                        x,y,w,h = t[cy, cx, 1:5].tolist()
                        gt.append((cls, xywh_to_xyxy(x,y,w,h)))

            matched = set()
            for pc, pconf, px1,py1,px2,py2 in pred_list:
                best_iou = 0.0
                best_j = -1
                for j,(gc,gbox) in enumerate(gt):
                    if j in matched:
                        continue
                    if pc != gc:
                        continue
                    iou = iou_xyxy((px1,py1,px2,py2), gbox)
                    if iou > best_iou:
                        best_iou = iou
                        best_j = j
                if best_iou >= iou_thresh and best_j >= 0:
                    tp += 1
                    matched.add(best_j)
                else:
                    fp += 1

            fn += (len(gt) - len(matched))

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1

def main():
    base = Path.cwd()

    data_yaml = base / "arhiva 1" / "car" / "data.yaml"
    data = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    names = data["names"]
    C = int(data["nc"])

    train_img = base / "arhiva 1" / "car" / "train" / "images"
    train_lbl = base / "arhiva 1" / "car" / "train" / "labels"
    val_img   = base / "arhiva 1" / "car" / "valid" / "images"
    val_lbl   = base / "arhiva 1" / "car" / "valid" / "labels"
    test_img  = base / "arhiva 1" / "car" / "test"  / "images"
    test_lbl  = base / "arhiva 1" / "car" / "test"  / "labels"

    IMG_SIZE = 416
    S = 13
    EPOCHS = 2     
    BATCH = 16
    LR = 1e-3

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_ds = YoloGridDataset(train_img, train_lbl, img_size=IMG_SIZE, S=S, C=C)
    val_ds   = YoloGridDataset(val_img, val_lbl,   img_size=IMG_SIZE, S=S, C=C)
    test_ds  = YoloGridDataset(test_img, test_lbl, img_size=IMG_SIZE, S=S, C=C)

    train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
    val_ld   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
    test_ld  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

    model = TinyDetector(C=C).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_f1 = -1.0
    out_dir = base / "scratch_runs"
    out_dir.mkdir(exist_ok=True)
    best_path = out_dir / "best_detector.pt"

    # -------- TRAIN ----------
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for imgs, targets in tqdm(train_ld, desc=f"Epoch {epoch}/{EPOCHS}"):
            imgs = imgs.to(device)
            targets = targets.to(device)

            pred = model(imgs)
            loss = detection_loss(pred, targets)

            opt.zero_grad()
            loss.backward()
            opt.step()

            losses.append(loss.item())

        avg_loss = float(np.mean(losses))

        # evaluare pe VAL la final de epocă
        p, r, f1 = evaluate_prf1(model, val_ld, device, C=C, iou_thresh=0.5, conf_thresh=0.25)
        print(f"Epoch {epoch}/{EPOCHS} | loss={avg_loss:.4f} | VAL P={p:.3f} R={r:.3f} F1={f1:.3f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), best_path)
            print("  ✅ saved best model ->", best_path)

    # -------- TEST ----------
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.to(device)
    p, r, f1 = evaluate_prf1(model, test_ld, device, C=C, iou_thresh=0.5, conf_thresh=0.25)
    print(f"\nFINAL TEST: P={p:.3f} R={r:.3f} F1={f1:.3f}")

    # -------- DEMO IMAGES ----------
    demo_dir = out_dir / "demo_preds"
    demo_dir.mkdir(exist_ok=True)
    sample_imgs = random.sample(test_ds.imgs, k=min(8, len(test_ds.imgs)))

    model.eval()
    for img_path in sample_imgs:
        img0 = cv2.imread(str(img_path))
        img0 = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img0, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
        x = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(x).cpu()[0]   # (S,S,5+C)

        dets = decode_predictions(pred, conf_thresh=0.10)
        if len(dets) > 0:
            boxes = np.array([[d[2],d[3],d[4],d[5]] for d in dets], dtype=np.float32)
            scores = np.array([d[1] for d in dets], dtype=np.float32)
            keep = nms(boxes, scores, iou_thresh=0.5)
            dets = [dets[i] for i in keep]

        drawn = draw_dets(cv2.resize(img0, (IMG_SIZE, IMG_SIZE)), dets, names)
        out_path = demo_dir / f"{img_path.stem}_pred.png"
        cv2.imwrite(str(out_path), cv2.cvtColor(drawn, cv2.COLOR_RGB2BGR))

    print("✅ Demo predictions saved in:", demo_dir.resolve())


In [27]:
main()


Device: cpu


Epoch 1/2: 100%|█████████████████████████████████████████████████████████████████████| 221/221 [16:41<00:00,  4.53s/it]
                                                                                                                       

Epoch 1/2 | loss=2.5919 | VAL P=0.000 R=0.000 F1=0.000
  ✅ saved best model -> C:\Users\Raul\Desktop\Facultate\An 3\Proiect SBC\3 - Autonomous Car Navigation System\scratch_runs\best_detector.pt


Epoch 2/2: 100%|█████████████████████████████████████████████████████████████████████| 221/221 [12:45<00:00,  3.46s/it]
                                                                                                                       

Epoch 2/2 | loss=2.2159 | VAL P=0.013 R=0.006 F1=0.008
  ✅ saved best model -> C:\Users\Raul\Desktop\Facultate\An 3\Proiect SBC\3 - Autonomous Car Navigation System\scratch_runs\best_detector.pt



FINAL TEST: P=0.011 R=0.007 F1=0.008
✅ Demo predictions saved in: C:\Users\Raul\Desktop\Facultate\An 3\Proiect SBC\3 - Autonomous Car Navigation System\scratch_runs\demo_preds


In [ ]:
import numpy as np
import torch
import cv2
from pathlib import Path
import random
from tqdm import tqdm

def xywh_to_xyxy(x, y, w, h):
    x1 = x - w/2
    y1 = y - h/2
    x2 = x + w/2
    y2 = y + h/2
    return x1, y1, x2, y2

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    union = area_a + area_b - inter + 1e-9
    return inter / union

def nms(boxes, scores, iou_thresh=0.5):
    idxs = np.argsort(scores)[::-1]
    keep = []
    while len(idxs) > 0:
        i = int(idxs[0])
        keep.append(i)
        rest = idxs[1:]
        new_rest = []
        for j in rest:
            if iou_xyxy(boxes[i], boxes[int(j)]) < iou_thresh:
                new_rest.append(int(j))
        idxs = np.array(new_rest, dtype=np.int64)
    return keep

@torch.no_grad()
def decode_predictions(pred, conf_thresh=0.25):
    """
    pred: (S,S,5+C) raw
    return: list (cls, conf, x1,y1,x2,y2) normalized in [0,1]
    """
    S = pred.shape[0]
    C = pred.shape[-1] - 5

    obj = torch.sigmoid(pred[..., 0])
    box = torch.sigmoid(pred[..., 1:5])
    cls_prob = torch.softmax(pred[..., 5:], dim=-1)

    out = []
    for cy in range(S):
        for cx in range(S):
            o = float(obj[cy, cx].item())
            if o < conf_thresh:
                continue
            c = int(torch.argmax(cls_prob[cy, cx]).item())
            p = float(cls_prob[cy, cx, c].item())
            conf = o * p

            x, y, w, h = box[cy, cx].tolist()
            x1, y1, x2, y2 = xywh_to_xyxy(x, y, w, h)
            
            x1 = max(0.0, min(1.0, x1))
            y1 = max(0.0, min(1.0, y1))
            x2 = max(0.0, min(1.0, x2))
            y2 = max(0.0, min(1.0, y2))
            out.append((c, conf, x1, y1, x2, y2))

    return out

def draw_dets(img_rgb, dets, names):
    h, w = img_rgb.shape[:2]
    img = img_rgb.copy()
    for c, conf, x1,y1,x2,y2 in dets:
        X1 = int(x1*w); Y1 = int(y1*h); X2 = int(x2*w); Y2 = int(y2*h)
        cv2.rectangle(img, (X1, Y1), (X2, Y2), (0,255,0), 2)
        txt = f"{names[c]} {conf:.2f}"
        cv2.putText(img, txt, (X1, max(15, Y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    return img

@torch.no_grad()
def evaluate_prf1(model, loader, device, names, iou_thresh=0.5, conf_thresh=0.25):
    """
    Metric simplu:
    TP = pred (clasă corectă) cu IoU>=thr cu un GT nematcheduit
    FP = pred fără match
    FN = GT nematcheduit
    """
    model.eval()
    tp = fp = fn = 0

    for imgs, targets in tqdm(loader, desc="Eval", leave=False):
        imgs = imgs.to(device)
        preds = model(imgs).cpu()

        B = preds.shape[0]
        S = preds.shape[1]

        for b in range(B):
            pred_list = decode_predictions(preds[b], conf_thresh=conf_thresh)

            # NMS
            if len(pred_list) > 0:
                boxes = np.array([[p[2],p[3],p[4],p[5]] for p in pred_list], dtype=np.float32)
                scores = np.array([p[1] for p in pred_list], dtype=np.float32)
                keep = nms(boxes, scores, iou_thresh=0.5)
                pred_list = [pred_list[i] for i in keep]

            # GT din grid
            gt = []
            t = targets[b]
            for cy in range(S):
                for cx in range(S):
                    if t[cy, cx, 0].item() == 1.0:
                        cls = int(torch.argmax(t[cy, cx, 5:]).item())
                        x,y,w,h = t[cy, cx, 1:5].tolist()
                        gt.append((cls, xywh_to_xyxy(x,y,w,h)))

            matched = set()
            for pc, pconf, px1,py1,px2,py2 in pred_list:
                best_iou = 0.0
                best_j = -1
                for j,(gc,gbox) in enumerate(gt):
                    if j in matched: 
                        continue
                    if pc != gc:
                        continue
                    iou = iou_xyxy((px1,py1,px2,py2), gbox)
                    if iou > best_iou:
                        best_iou = iou
                        best_j = j
                if best_iou >= iou_thresh and best_j >= 0:
                    tp += 1
                    matched.add(best_j)
                else:
                    fp += 1

            fn += (len(gt) - len(matched))

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1


In [29]:
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm

def main_train(EPOCHS=5, BATCH=16, LR=1e-3, IMG_SIZE=416, S=13):
    base = Path.cwd()

    data_yaml = base / "arhiva 1" / "car" / "data.yaml"
    data = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    names = data["names"]
    C = int(data["nc"])

    train_img = base / "arhiva 1" / "car" / "train" / "images"
    train_lbl = base / "arhiva 1" / "car" / "train" / "labels"
    val_img   = base / "arhiva 1" / "car" / "valid" / "images"
    val_lbl   = base / "arhiva 1" / "car" / "valid" / "labels"
    test_img  = base / "arhiva 1" / "car" / "test" / "images"
    test_lbl  = base / "arhiva 1" / "car" / "test" / "labels"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_ds = YoloGridDataset(train_img, train_lbl, img_size=IMG_SIZE, S=S, C=C)
    val_ds   = YoloGridDataset(val_img,   val_lbl,   img_size=IMG_SIZE, S=S, C=C)
    test_ds  = YoloGridDataset(test_img,  test_lbl,  img_size=IMG_SIZE, S=S, C=C)

    train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
    val_ld   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
    test_ld  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

    model = TinyDetector(C=C).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for imgs, targets in tqdm(train_ld, desc=f"Epoch {epoch}/{EPOCHS}"):
            imgs = imgs.to(device)
            targets = targets.to(device)

            pred = model(imgs)
            loss = detection_loss(pred, targets)

            opt.zero_grad()
            loss.backward()
            opt.step()

            losses.append(loss.item())

        print(f"Epoch {epoch}/{EPOCHS} - Loss: {float(np.mean(losses)):.4f}")

    return model, device, names, val_ld, test_ld, IMG_SIZE


In [ ]:
model, device, names, val_ld, test_ld, IMG_SIZE = main_train(EPOCHS=2)  

Path("scratch_runs").mkdir(exist_ok=True)
torch.save(model.state_dict(), "scratch_runs/model_last.pt")
print("✅ Model salvat: scratch_runs/model_last.pt")


Device: cpu


Epoch 1/2: 100%|█████████████████████████████████████████████████████████████████████| 221/221 [09:05<00:00,  2.47s/it]


Epoch 1/2 - Loss: 2.6512


Epoch 2/2: 100%|█████████████████████████████████████████████████████████████████████| 221/221 [09:03<00:00,  2.46s/it]

Epoch 2/2 - Loss: 2.4115
✅ Model salvat: scratch_runs/model_last.pt


In [31]:
p, r, f1 = evaluate_prf1(model, val_ld, device, names, iou_thresh=0.5, conf_thresh=0.25)
print(f"VALID: Precision={p:.3f} Recall={r:.3f} F1={f1:.3f}")

p, r, f1 = evaluate_prf1(model, test_ld, device, names, iou_thresh=0.5, conf_thresh=0.25)
print(f"TEST:  Precision={p:.3f} Recall={r:.3f} F1={f1:.3f}")


VALID: Precision=0.000 Recall=0.000 F1=0.000


TEST:  Precision=0.000 Recall=0.000 F1=0.000


In [32]:
demo_dir = Path("scratch_runs/demo_preds")
demo_dir.mkdir(parents=True, exist_ok=True)

test_images_dir = Path.cwd() / "arhiva 1" / "car" / "test" / "images"
test_imgs = sorted([p for p in test_images_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
sample = random.sample(test_imgs, k=min(8, len(test_imgs)))

model.eval()
for img_path in sample:
    img0 = cv2.imread(str(img_path))
    img0 = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img0, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0

    x = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x).cpu()[0]  # (S,S,5+C)

    dets = decode_predictions(pred, conf_thresh=0.10)

    # NMS
    if len(dets) > 0:
        boxes = np.array([[d[2],d[3],d[4],d[5]] for d in dets], dtype=np.float32)
        scores = np.array([d[1] for d in dets], dtype=np.float32)
        keep = nms(boxes, scores, iou_thresh=0.5)
        dets = [dets[i] for i in keep]

    drawn = draw_dets(cv2.resize(img0, (IMG_SIZE, IMG_SIZE)), dets, names)
    out_path = demo_dir / f"{img_path.stem}_pred.png"
    cv2.imwrite(str(out_path), cv2.cvtColor(drawn, cv2.COLOR_RGB2BGR))

print("✅ Gata! Vezi imaginile în:", demo_dir.resolve())


✅ Gata! Vezi imaginile în: C:\Users\Raul\Desktop\Facultate\An 3\Proiect SBC\3 - Autonomous Car Navigation System\scratch_runs\demo_preds
